# Phase 11B — Statistical comparisons and calibration from locked grouped-CV predictions

## Kaggle inputs

Attach **exactly these two saved outputs**, as Kaggle datasets or notebook outputs (ZIP files and Kaggle-unpacked contents both work):

1. `[2.1] enhancer_phase2_corrected_results.zip`
2. `3_enhancer_phase3_robustness.zip`

**CPU only; Internet off; no checkpoint, Phase 1 FASTA, or Phase 11A ZIP required.** Save and download `enhancer_phase11b_statistics_calibration.zip` from Kaggle Output and share it here. This notebook is analysis only; it does not fit new models or overwrite earlier results.

It compares four fixed contrasts: DNABERT-2 minus spaced 6/4-mer, and DNABERT-2 minus legacy DNABERT, at each of Stages 1 and 2. These contrasts were specified for this notebook **after earlier Phase 2/3 results had been examined**. Holm correction is applied to the four exploratory tests. Nadeau–Bengio corrected paired fold *t* tests are shown as sensitivity analyses with only five folds, not definitive proof. Phase 3's existing similarity-group bootstrap replicates provide conditional simultaneous 95% intervals; they do not capture new training seeds. The formula follows the original [Nadeau and Bengio work](https://papers.neurips.cc/paper/1999/file/7d12b66d3df6af8d429c1a357d8b9e1a-Paper.pdf).

Calibration, Brier score, 10-bin ECE, reliability diagrams, confusion matrices, and sensitivity/specificity/MCC are calculated from the existing **out of fold** predictions. Threshold 0.5 is reported as an explicitly fixed descriptive operating point. **This notebook cannot select a legitimate inner-validation threshold from the saved outer-fold predictions** because those other fold predictions came from models whose training included the target outer fold. Proper inner-validation threshold selection requires new within-training-fold fits; reserve that for Phase 11C.

In [1]:
# 0 — Imports and fully specified analysis lock.
import os,io,re,json,zipfile,hashlib,platform
from pathlib import Path
import numpy as np,pandas as pd,scipy,sklearn
from scipy.stats import t as student_t
from sklearn.metrics import (roc_auc_score,average_precision_score,brier_score_loss,
 accuracy_score,balanced_accuracy_score,matthews_corrcoef,confusion_matrix)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
INPUT=Path('/kaggle/input')
assert INPUT.is_dir(),'Attach the Phase 2.1 and Phase 3 saved Outputs listed above.'
WORK=Path('/kaggle/working/enhancer_phase11b_work');WORK.mkdir(parents=True,exist_ok=True)
LOCKED={
 'correction_report.json':'8296e5703c8e73765b42923be970b6752061639331c1318c6a3414767ed6914a',
 'all_oof_predictions.csv':'08d5d685690ec27bb36fdf6e7bbf8e3ff8dc7619484f9b3b51be54ff213e871f',
 'fold_metrics.csv':'00f55acbac796f066cdab89b00379c7d0b18464a21b16400fff4e106be14b8da',
 'phase3_run_report.json':'e6c91050f7b9beedde686d11484cbe5ca30fb462c7c748c2f68cea4119626049',
 'phase3_bootstrap_replicates.npz':'574348387685b603a17533e82e18c47942e7a4517842d5422b0f11d70cfbabae',
 'phase3_paired_bootstrap_intervals.csv':'5b53b62f470bfca0170ffb41e6909db3c9e9e392354288ed2e694e54dd8fa5bc'}
CONFIG={'name':'phase11b_oof_statistics_calibration_v1','input_member_sha256':LOCKED,
 'phase2_config_hash':'63e47ce62c02127cc6bf71762c1205d3bf473cc84085cae3f455ceffb7b1e5a3',
 'phase2_archive_sha256':'332549354d3e41ad0d1e968ad3d4db78ca55beaa47cc3c0a53422ccfec0a6c83',
 'phase3_archive_sha256':'775aae5374e849b8ce9e33f7fabe045f9d4b684f70bc190b9bde85cd5da4734c',
 'comparisons':[('finetuned_dnabert2','spaced6_4_logreg'),
                ('finetuned_dnabert2','finetuned_dnabert_legacy')],
 'stages':[1,2],'metric':'fold AUROC','holm_family_size':4,
 'bootstrap_source':'Phase 3 same-group within-fold resamples; 1000 fixed-model replicates',
 'ece_bins':10,'threshold':0.5,'threshold_grid':[round(v,2) for v in np.linspace(.05,.95,19)]}
def sha(b):return hashlib.sha256(b).hexdigest()
PROTOCOL=sha(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode())
print('Phase 11B protocol',PROTOCOL)

Phase 11B protocol a7ac0a2e5c4ee74c0e7130bcbedfa264b3b80f0f35f6c45b777da0287e5e2392


In [2]:
# 1 — Read authenticated members from zipped or Kaggle-unpacked Inputs.
# Multiple identical copies are safe; conflicting versions never silently win.
ZIPS=sorted(p for p in INPUT.rglob('*.zip') if p.is_file())
def read_member(name):
 hits=[];mismatches=[]
 for p in INPUT.rglob(name):
  if p.is_file():
   raw=p.read_bytes()
   (hits if sha(raw)==LOCKED[name] else mismatches).append((raw,str(p)))
 for p in ZIPS:
  try:
   with zipfile.ZipFile(p) as z:
    matches=[x for x in z.namelist() if x==name or x.endswith('/'+name)]
    for member in matches:
     raw=z.read(member)
     (hits if sha(raw)==LOCKED[name] else mismatches).append((raw,str(p)+'!'+member))
  except (OSError,zipfile.BadZipFile):continue
 if not hits:
  raise FileNotFoundError(f'Missing or wrong-version {name}. Found unmatched: {[m[1] for m in mismatches[:8]]}. Attach Phase 2.1 and Phase 3 saved Outputs.')
 assert len({sha(raw) for raw,_ in hits})==1
 print('VERIFIED',name,'from',hits[0][1]);return hits[0][0]
RAW={n:read_member(n) for n in LOCKED}
corr=json.loads(RAW['correction_report.json'])
report=json.loads(RAW['phase3_run_report.json'])
assert corr['config_hash']==CONFIG['phase2_config_hash']
assert report['phase2_config_hash']==CONFIG['phase2_config_hash']
assert report['phase2_1_zip_sha256']==CONFIG['phase2_archive_sha256']
for n in ('all_oof_predictions.csv','fold_metrics.csv'):
 assert corr['output_sha256'][n]==LOCKED[n]
for n in ('phase3_bootstrap_replicates.npz','phase3_paired_bootstrap_intervals.csv'):
 assert report['output_sha256'][n]==LOCKED[n]
assert report['bootstrap_replicates']==1000
pred=pd.read_csv(io.BytesIO(RAW['all_oof_predictions.csv']),dtype={'record_id':str,'similarity_group_id':str,'model':str,'config_hash':str})
old_fold=pd.read_csv(io.BytesIO(RAW['fold_metrics.csv']))
phase3_interval=pd.read_csv(io.BytesIO(RAW['phase3_paired_bootstrap_intervals.csv']))
with np.load(io.BytesIO(RAW['phase3_bootstrap_replicates.npz']),allow_pickle=False) as npz:
 boot={k:npz[k].copy() for k in npz.files}
assert len(pred)==37926 and set(pred.stage)=={1,2}
assert pred.config_hash.eq(CONFIG['phase2_config_hash']).all()
assert not pred.duplicated(['stage','model','record_id']).any()
assert pred.score.between(0,1).all() and np.isfinite(pred.score).all()
assert set(pred.fold_5)==set(range(5)) and set(pred.label)=={0,1}
assert pred.groupby(['stage','similarity_group_id']).fold_5.nunique().max()==1
assert len(boot['model'])==4 and set(boot['stage'])=={1,2}
assert boot['raw_mean_fold'].shape==(1000,2,4)
assert boot['seed'].tolist()==[20260928]
for stage in (1,2):
 truth=pred[(pred.stage==stage)&(pred.model=='finetuned_dnabert2')].set_index('record_id')
 for model,part in pred[pred.stage==stage].groupby('model'):
  assert len(part)==len(truth) and set(part.record_id)==set(truth.index)
  expected=truth.loc[part.record_id]
  for col in ('label','fold_5','similarity_group_id'):
   assert part[col].tolist()==expected[col].tolist(),(stage,model,col)
print('All prediction IDs, labels, folds, groups and Phase 3 bootstrap inputs agree.')

VERIFIED correction_report.json from /kaggle/input/datasets/aryanaditta/enhancer-phase2-corrected-results-zip/correction_report.json
VERIFIED all_oof_predictions.csv from /kaggle/input/datasets/aryanaditta/enhancer-phase2-corrected-results-zip/all_oof_predictions.csv
VERIFIED fold_metrics.csv from /kaggle/input/datasets/aryanaditta/enhancer-phase2-corrected-results-zip/fold_metrics.csv
VERIFIED phase3_run_report.json from /kaggle/input/datasets/aryanaditta/3-enhancer-phase3-robustness-zip/phase3_run_report.json
VERIFIED phase3_bootstrap_replicates.npz from /kaggle/input/datasets/aryanaditta/3-enhancer-phase3-robustness-zip/phase3_bootstrap_replicates.npz
VERIFIED phase3_paired_bootstrap_intervals.csv from /kaggle/input/datasets/aryanaditta/3-enhancer-phase3-robustness-zip/phase3_paired_bootstrap_intervals.csv
All prediction IDs, labels, folds, groups and Phase 3 bootstrap inputs agree.


In [3]:
# 2 — Reproduce archived fold AUROCs before statistical comparisons.
CHECK_MODELS=set(sum((list(pair) for pair in CONFIG['comparisons']),[]))
fold=[]
for (stage,model,f),g in pred[pred.model.isin(CHECK_MODELS)].groupby(['stage','model','fold_5']):
 auc=roc_auc_score(g.label,g.score)
 saved=old_fold[(old_fold.stage==stage)&(old_fold.model==model)&(old_fold.fold_5==f)]
 assert len(saved)==1 and abs(auc-float(saved.roc_auc.iloc[0]))<1e-10
 fold.append({'stage':stage,'model':model,'fold_5':f,'n_validation':len(g),
              'n_training':len(pred[(pred.stage==stage)&(pred.model==model)])-len(g),
              'auroc':auc,'auprc':average_precision_score(g.label,g.score)})
fold=pd.DataFrame(fold)
fold.to_csv(WORK/'phase11b_fold_metrics_verified.csv',index=False)
print(fold.groupby(['stage','model']).auroc.agg(['mean','std']).to_string())

                                    mean       std
stage model                                       
1     finetuned_dnabert2        0.875778  0.010770
      finetuned_dnabert_legacy  0.872397  0.018180
      spaced6_4_logreg          0.851279  0.021298
2     finetuned_dnabert2        0.708626  0.018656
      finetuned_dnabert_legacy  0.690829  0.005865
      spaced6_4_logreg          0.706396  0.014369


### Statistical interpretation

The corrected paired fold test uses the five matched fold AUROC differences and

\[t=\bar d / \sqrt{(1/5 + n_{test}/n_{train})s_d^2},\qquad df=4.\]

`n_test/n_train` is the mean of each held-out fold size divided by its training size. The small sample of five dependent folds makes this an **approximation**. Holm correction applies across all four stage/contrast tests selected for this notebook. The separate simultaneous intervals reuse the *same* Phase 3 group bootstrap replicates, with a max standardized absolute deviation across these four contrasts; they are **conditional on the existing fitted models and fixed folds**. Neither method estimates seed-to-seed training variation, to be evaluated in Phase 11C.

In [4]:
# 3 — Corrected paired-fold t tests and Holm correction over the four fixed comparisons.
from scipy.stats import t as t_distribution
rows=[]
for stage in CONFIG['stages']:
 sub=fold[fold.stage==stage]
 wide=sub.pivot(index='fold_5',columns='model',values='auroc')
 sizes=sub[sub.model=='finetuned_dnabert2'].sort_values('fold_5')
 ratio=float(np.mean(sizes.n_validation.to_numpy()/sizes.n_training.to_numpy()))
 for a,b in CONFIG['comparisons']:
  d=(wide[a]-wide[b]).to_numpy();k=len(d)
  assert k==5
  sd=float(np.std(d,ddof=1));factor=1/k+ratio
  se=float(np.sqrt(factor)*sd)
  tval=float(d.mean()/se) if se>0 else (np.inf if d.mean()>0 else -np.inf if d.mean()<0 else 0.)
  p=float(2*t_distribution.sf(abs(tval),df=k-1))
  crit=float(t_distribution.ppf(.975,df=k-1))
  rows.append({'stage':stage,'model_a':a,'model_b':b,'mean_fold_auc_delta':float(d.mean()),
   'fold_delta_sd':sd,'validation_to_training_ratio':ratio,'correction_factor':factor,
   'corrected_standard_error':se,'corrected_t':tval,'df':k-1,'raw_two_sided_p':p,
   'approx_unadjusted_ci_low':float(d.mean()-crit*se),
   'approx_unadjusted_ci_high':float(d.mean()+crit*se)})
stat=pd.DataFrame(rows)
order=np.argsort(stat.raw_two_sided_p.to_numpy())
pvals=stat.raw_two_sided_p.to_numpy();adj=np.empty(len(pvals));max_seen=0.
for rank,ix in enumerate(order):
 max_seen=max(max_seen,min(1.,(len(pvals)-rank)*pvals[ix]))
 adj[ix]=max_seen
stat['holm_adjusted_p']=adj
stat['exploratory_family']='four stage-by-contrast tests fixed after prior Phase 2/3 inspection'
stat.to_csv(WORK/'phase11b_corrected_fold_tests.csv',index=False)
print(stat[['stage','model_a','model_b','mean_fold_auc_delta','corrected_t','raw_two_sided_p','holm_adjusted_p']].to_string(index=False))

 stage            model_a                  model_b  mean_fold_auc_delta  corrected_t  raw_two_sided_p  holm_adjusted_p
     1 finetuned_dnabert2         spaced6_4_logreg             0.024499     2.832916         0.047205         0.188821
     1 finetuned_dnabert2 finetuned_dnabert_legacy             0.003381     0.448622         0.676934         1.000000
     2 finetuned_dnabert2         spaced6_4_logreg             0.002230     0.190891         0.857908         1.000000
     2 finetuned_dnabert2 finetuned_dnabert_legacy             0.017797     1.577635         0.189786         0.569358


In [5]:
# 4 — Conditional simultaneous intervals from existing Phase 3 within-fold group bootstrap.
# Standardized max absolute deviation respects dependence among the four contrasts.
B=boot['raw_mean_fold'];models=list(boot['model']);stages=list(boot['stage'])
assert tuple(stages)==(1,2) and np.isfinite(B).all()
rep=[];identities=[]
for row in stat.itertuples(index=False):
 si=stages.index(row.stage);ia=models.index(row.model_a);ib=models.index(row.model_b)
 vals=B[:,si,ia]-B[:,si,ib]
 rep.append(vals)
 identities.append((row.stage,row.model_a,row.model_b))
R=np.column_stack(rep)
centers=stat.mean_fold_auc_delta.to_numpy()
sds=R.std(axis=0,ddof=1)
assert np.all(sds>0)
maxdev=np.max(np.abs((R-centers)/sds),axis=1)
critical=float(np.quantile(maxdev,.95))
intervals=stat[['stage','model_a','model_b','mean_fold_auc_delta']].copy()
intervals['conditional_simultaneous_95_low']=centers-critical*sds
intervals['conditional_simultaneous_95_high']=centers+critical*sds
intervals['bootstrap_sd']=sds
intervals['max_standardized_abs_critical']=critical
intervals['bootstrap_repeats']=len(R)
intervals['scope']='fixed-model within-fold similarity-group resampling; four-contrast simultaneous interval'
for x in intervals.itertuples():
 original=phase3_interval[(phase3_interval.stage==x.stage)&
  (phase3_interval.model_a==x.model_a)&(phase3_interval.model_b==x.model_b)&
  (phase3_interval.endpoint=='raw_mean_fold')]
 assert len(original)==1
 assert abs(x.mean_fold_auc_delta-float(original.observed_a_minus_b.iloc[0]))<1e-10
intervals.to_csv(WORK/'phase11b_simultaneous_conditional_intervals.csv',index=False)
print(intervals[['stage','model_a','model_b','mean_fold_auc_delta',
 'conditional_simultaneous_95_low','conditional_simultaneous_95_high']].to_string(index=False))

 stage            model_a                  model_b  mean_fold_auc_delta  conditional_simultaneous_95_low  conditional_simultaneous_95_high
     1 finetuned_dnabert2         spaced6_4_logreg             0.024499                         0.013187                          0.035811
     1 finetuned_dnabert2 finetuned_dnabert_legacy             0.003381                        -0.006113                          0.012875
     2 finetuned_dnabert2         spaced6_4_logreg             0.002230                        -0.024308                          0.028769
     2 finetuned_dnabert2 finetuned_dnabert_legacy             0.017797                        -0.008141                          0.043734


In [6]:
# 5 — Calibration and the fixed, prespecified 0.5 decision point.
MODELS=['gc_logreg','spaced6_4_logreg','finetuned_dnabert_legacy','finetuned_dnabert2']
NUM_BINS=CONFIG['ece_bins']
def ece_and_bins(g,stage,model,fold_name):
 y=g.label.to_numpy(dtype=int);s=g.score.to_numpy(dtype=float)
 bins=np.minimum(np.floor(s*NUM_BINS).astype(int),NUM_BINS-1)
 out=[]
 for b in range(NUM_BINS):
  ix=np.flatnonzero(bins==b)
  out.append({'stage':stage,'model':model,'fold':fold_name,'bin':b,
   'bin_low':b/NUM_BINS,'bin_high':(b+1)/NUM_BINS,'n':len(ix),
   'mean_score':float(s[ix].mean()) if len(ix) else np.nan,
   'observed_fraction':float(y[ix].mean()) if len(ix) else np.nan})
 populated=[r for r in out if r['n']]
 ece=sum(r['n']*abs(r['mean_score']-r['observed_fraction']) for r in populated)/len(y)
 return ece,out
cal=[];reliability=[];fixed=[]
for (stage,model),all_g in pred[pred.model.isin(MODELS)].groupby(['stage','model']):
 for fold_name,g in [('pooled_oof',all_g)]+[(str(f),part) for f,part in all_g.groupby('fold_5')]:
  y=g.label.to_numpy(dtype=int);s=g.score.to_numpy(dtype=float)
  ece,bins=ece_and_bins(g,stage,model,fold_name);reliability.extend(bins)
  cal.append({'stage':stage,'model':model,'fold':fold_name,'n':len(g),'positive':int(y.sum()),
   'brier':float(brier_score_loss(y,s)),'ece_10_equal_width':ece,
   'mean_score':float(s.mean()),'observed_prevalence':float(y.mean()),
   'auroc':float(roc_auc_score(y,s)),'auprc':float(average_precision_score(y,s))})
  hard=(s>=.5).astype(int);tn,fp,fn,tp=confusion_matrix(y,hard,labels=[0,1]).ravel()
  fixed.append({'stage':stage,'model':model,'fold':fold_name,'threshold':.5,
   'n':len(y),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp),
   'sensitivity':float(tp/(tp+fn)),'specificity':float(tn/(tn+fp)),
   'accuracy':float(accuracy_score(y,hard)),
   'balanced_accuracy':float(balanced_accuracy_score(y,hard)),
   'mcc':float(matthews_corrcoef(y,hard))})
cal=pd.DataFrame(cal);reliability=pd.DataFrame(reliability);fixed=pd.DataFrame(fixed)
for x in cal.itertuples(index=False):
 if x.fold!='pooled_oof':
  old=old_fold[(old_fold.stage==x.stage)&(old_fold.model==x.model)&
               (old_fold.fold_5==int(x.fold))]
  assert len(old)==1 and abs(x.brier-old.brier.iloc[0])<1e-10
for name,data in [('phase11b_calibration_metrics.csv',cal),
                  ('phase11b_reliability_bins.csv',reliability),
                  ('phase11b_fixed_0_5_metrics.csv',fixed)]:
 data.to_csv(WORK/name,index=False)
print('Pooled calibration:')
print(cal[cal.fold=='pooled_oof'][['stage','model','n','brier','ece_10_equal_width','mean_score','observed_prevalence']].to_string(index=False))
print('Pooled fixed threshold:')
print(fixed[fixed.fold=='pooled_oof'][['stage','model','tn','fp','fn','tp','sensitivity','specificity','balanced_accuracy','mcc']].to_string(index=False))

Pooled calibration:
 stage                    model    n    brier  ece_10_equal_width  mean_score  observed_prevalence
     1       finetuned_dnabert2 2838 0.169976            0.087615    0.517641             0.484848
     1 finetuned_dnabert_legacy 2838 0.149043            0.039978    0.496438             0.484848
     1                gc_logreg 2838 0.166596            0.020260    0.484653             0.484848
     1         spaced6_4_logreg 2838 0.156204            0.021610    0.485363             0.484848
     2       finetuned_dnabert2 1376 0.261910            0.173088    0.327744             0.465843
     2 finetuned_dnabert_legacy 1376 0.228705            0.056193    0.447657             0.465843
     2                gc_logreg 1376 0.230933            0.039841    0.465680             0.465843
     2         spaced6_4_logreg 1376 0.216727            0.017083    0.465210             0.465843
Pooled fixed threshold:
 stage                    model   tn  fp  fn   tp  sensitivity  s

In [7]:
# 6 — Descriptive threshold sweep only. These curves do not select or endorse any threshold.
threshold_rows=[]
for (stage,model),g in pred[pred.model.isin(MODELS)].groupby(['stage','model']):
 y=g.label.to_numpy(dtype=int);s=g.score.to_numpy(dtype=float)
 for threshold in CONFIG['threshold_grid']:
  hard=(s>=threshold).astype(int)
  tn,fp,fn,tp=confusion_matrix(y,hard,labels=[0,1]).ravel()
  threshold_rows.append({'stage':stage,'model':model,'threshold':threshold,
   'sensitivity':float(tp/(tp+fn)),'specificity':float(tn/(tn+fp)),
   'balanced_accuracy':float(balanced_accuracy_score(y,hard)),
   'mcc':float(matthews_corrcoef(y,hard)),
   'use':'descriptive sweep; no threshold selected'})
thresholds=pd.DataFrame(threshold_rows)
thresholds.to_csv(WORK/'phase11b_descriptive_threshold_curves.csv',index=False)
requirements={
 'status':'not computed in Phase 11B',
 'reason':'Other outer-fold OOF predictions were produced by training on the target outer fold; selecting its threshold from them is not nested validation.',
 'needed_for_valid_nested_thresholds':'For each outer held-out fold, fit one or more inner models using only that outer fold training partition, choose the threshold on inner validation predictions, and apply it without modification to outer held-out predictions.',
 'proposed_selection_metric':'balanced accuracy (fixed before inner fitting)',
 'tie_break':'closest to 0.5, then lower threshold',
 'phase11c_note':'If comparing training seeds, keep inner threshold selection isolated within each seed and outer fold.'}
(WORK/'inner_validation_threshold_requirement.json').write_text(json.dumps(requirements,indent=2)+'\n')
print('Threshold sweep exported for description; no best threshold is selected.')

Threshold sweep exported for description; no best threshold is selected.


In [8]:
# 7 — Figures and independently checkable output archive.
fig_files=[]
for stage in (1,2):
 fig,ax=plt.subplots(figsize=(6.8,5.5))
 ax.plot([0,1],[0,1],color='black',linestyle='--',linewidth=1,label='ideal')
 for model in MODELS:
  b=reliability[(reliability.stage==stage)&(reliability.model==model)&
                (reliability.fold=='pooled_oof')&(reliability.n>0)]
  ax.plot(b.mean_score,b.observed_fraction,marker='o',label=model)
 ax.set(xlim=(0,1),ylim=(0,1),xlabel='Mean predicted probability',
        ylabel='Observed positive fraction',title=f'Stage {stage}: out of fold calibration')
 ax.legend(fontsize=8);ax.grid(alpha=.2);fig.tight_layout()
 name=f'phase11b_stage{stage}_reliability.png';fig.savefig(WORK/name,dpi=200);fig_files.append(name);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4.5))
for model in MODELS:
 b=thresholds[(thresholds.stage==2)&(thresholds.model==model)]
 ax.plot(b.threshold,b.sensitivity,marker='.',label=model+' sensitivity')
ax.axvline(.5,color='black',linestyle='--',linewidth=1)
ax.set(xlabel='Decision threshold (descriptive only)',ylabel='Stage 2 sensitivity',ylim=(0,1))
ax.legend(fontsize=7);ax.grid(alpha=.2);fig.tight_layout()
name='phase11b_stage2_threshold_sensitivity.png';fig.savefig(WORK/name,dpi=200);fig_files.append(name);plt.close(fig)
fig,ax=plt.subplots(figsize=(8.5,4.5))
order=intervals.reset_index(drop=True)
ypos=np.arange(len(order))
ax.errorbar(order.mean_fold_auc_delta,ypos,
 xerr=[order.mean_fold_auc_delta-order.conditional_simultaneous_95_low,
       order.conditional_simultaneous_95_high-order.mean_fold_auc_delta],
 fmt='o',capsize=4)
ax.axvline(0,color='black',linewidth=1)
ax.set_yticks(ypos,[f'S{r.stage}: {r.model_a} − {r.model_b}' for r in order.itertuples()])
ax.set_xlabel('Mean fold AUROC difference; conditional simultaneous 95% intervals')
ax.grid(axis='x',alpha=.2);fig.tight_layout()
name='phase11b_conditional_auc_contrasts.png';fig.savefig(WORK/name,dpi=200);fig_files.append(name);plt.close(fig)
readme=(
 'Phase 11B: statistics and calibration from frozen Phase 2.1 grouped-CV OOF predictions.\n'
 'Four comparisons were fixed for this notebook after prior Phase 2/3 inspection; p values remain exploratory.\n'
 'Corrected fold t tests use only five dependent folds and are approximate.\n'
 'Simultaneous group bootstrap intervals use Phase 3 fixed-model predictions; they exclude retraining uncertainty.\n'
 'Reliability, ECE, Brier and 0.5 operating point are descriptive OOF estimates; ECE depends on binning.\n'
 'No nested-validation-selected threshold is estimated by this analysis.\n')
(WORK/'README_INTERPRETATION.txt').write_text(readme)
files=['phase11b_fold_metrics_verified.csv','phase11b_corrected_fold_tests.csv',
 'phase11b_simultaneous_conditional_intervals.csv','phase11b_calibration_metrics.csv',
 'phase11b_reliability_bins.csv','phase11b_fixed_0_5_metrics.csv',
 'phase11b_descriptive_threshold_curves.csv','inner_validation_threshold_requirement.json',
 'README_INTERPRETATION.txt']+fig_files
provenance={'protocol_sha256':PROTOCOL,'config':CONFIG,
 'software':{'python':platform.python_version(),'numpy':np.__version__,
 'pandas':pd.__version__,'scipy':scipy.__version__,'scikit_learn':sklearn.__version__},
 'output_sha256':{n:sha((WORK/n).read_bytes()) for n in files},
 'limitations':['Existing saved models, no new training seeds or prospective independent test.',
 'Contrasts fixed after prior results were examined.',
 'Five-fold corrected t approximation and fixed-model group bootstrap differ in uncertainty target.',
 'No threshold has been selected on a proper inner validation partition.']}
(WORK/'phase11b_provenance.json').write_text(json.dumps(provenance,indent=2,sort_keys=True)+'\n')
output=Path('/kaggle/working/enhancer_phase11b_statistics_calibration.zip')
with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for n in files+['phase11b_provenance.json']:z.write(WORK/n,n)
with zipfile.ZipFile(output) as z:
 assert z.testzip() is None
 for n,h in provenance['output_sha256'].items():assert sha(z.read(n))==h
print('DONE',output,'SHA256',sha(output.read_bytes()),'files',len(files)+1)

DONE /kaggle/working/enhancer_phase11b_statistics_calibration.zip SHA256 e9260e8c3e142c114010891bcc4fcab989cf151af60a13b3b2e03e6d5dcf811e files 14
